In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU name: Tesla T4


In [2]:
!pip install -q -U transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.1 MB/s eta 0:00:00


### Importing dataset and version verificaiton

In [6]:
import torch
import transformers
import datasets
import evaluate
import accelerate

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)
print("Evaluate version:", evaluate.__version__)
print("Accelerate version:", accelerate.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
Transformers version: 5.14.1
Datasets version: 5.0.0
Evaluate version: 0.4.6
Accelerate version: 1.14.0
GPU available: True


## Loading the SQuAD dataset

In [7]:
from datasets import load_dataset

# Load the Stanford Question Answering Dataset
squad = load_dataset("rajpurkar/squad")

print(squad)

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


## Inspecting one SQuAD eg

In [8]:
sample = squad["train"][0]

print("Title:")
print(sample["title"])

print("\nQuestion:")
print(sample["question"])

print("\nContext:")
print(sample["context"])

print("\nCorrect answer:")
print(sample["answers"]["text"][0])

print("\nAnswer starts at character position:")
print(sample["answers"]["answer_start"][0])

Title:
University_of_Notre_Dame

Question:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Context:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

Correct answer:
Saint Bernadette Soubirous

Answer starts at character position:
515


### creating training and validation subsets

In [9]:
# Create reproducible, shuffled subsets
train_dataset = (
    squad["train"]
    .shuffle(seed=42)
    .select(range(10000))
)

validation_dataset = (
    squad["validation"]
    .shuffle(seed=42)
    .select(range(2000))
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))

Training examples: 10000
Validation examples: 2000


## Loading DistilBERT tokenzier

In [10]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint,
    use_fast=True
)

print("Tokenizer loaded successfully")
print("Tokenizer type:", type(tokenizer).__name__)
print("Maximum supported length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully
Tokenizer type: BertTokenizer
Maximum supported length: 512


Tokenizer will convert the SQuAD questoins and context paragraphs into token IDs that DistillBERT can process. A fast tokenizer is needed as its character-to-token positino mapping is needed to find appropriate answer from the given context.

### tokenization validation

In [11]:
example = train_dataset[0]

encoded_example = tokenizer(
    example["question"].strip(),
    example["context"],
    max_length=384,
    truncation="only_second",
    padding="max_length",
    return_offsets_mapping=True
)

print("Fast tokenizer:", tokenizer.is_fast)
print("Question:", example["question"])
print("Correct answer:", example["answers"]["text"][0])

print("\nNumber of tokens:", len(encoded_example["input_ids"]))

print("\nFirst 20 tokens:")
print(
    tokenizer.convert_ids_to_tokens(
        encoded_example["input_ids"][:20]
    )
)

print("\nFirst 20 token IDs:")
print(encoded_example["input_ids"][:20])

Fast tokenizer: True
Question: What percentage of Egyptians polled support death penalty for those leaving Islam?
Correct answer: 84%

Number of tokens: 384

First 20 tokens:
['[CLS]', 'what', 'percentage', 'of', 'egyptians', 'polled', 'support', 'death', 'penalty', 'for', 'those', 'leaving', 'islam', '?', '[SEP]', 'the', 'pew', 'forum', 'on', 'religion']

First 20 token IDs:
[101, 2054, 7017, 1997, 23437, 26847, 2490, 2331, 6531, 2005, 2216, 2975, 7025, 1029, 102, 1996, 29071, 7057, 2006, 4676]


## Preprocessing function

In [12]:
max_length = 384


def preprocess_function(examples):
    # Remove unnecessary spaces from questions
    questions = [
        question.strip()
        for question in examples["question"]
    ]

    # Tokenize each question together with its context
    tokenized_inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    # Connect each token with its original character positions
    offset_mapping = tokenized_inputs.pop("offset_mapping")

    answers = examples["answers"]

    start_positions = []
    end_positions = []

    for example_index, offsets in enumerate(offset_mapping):
        answer = answers[example_index]

        # Character positions of the correct answer
        start_character = answer["answer_start"][0]
        answer_text = answer["text"][0]
        end_character = start_character + len(answer_text)

        # Identify question tokens and context tokens
        sequence_ids = tokenized_inputs.sequence_ids(example_index)

        # Find the first context token
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1

        # Find the final context token
        context_end = context_start
        while (
            context_end < len(sequence_ids)
            and sequence_ids[context_end] == 1
        ):
            context_end += 1

        context_end -= 1

        # Check whether truncation removed the answer
        if (
            offsets[context_start][0] > end_character
            or offsets[context_end][1] < start_character
        ):
            start_positions.append(0)
            end_positions.append(0)

        else:
            # Find the token containing the answer's first character
            token_start = context_start

            while (
                token_start <= context_end
                and offsets[token_start][0] <= start_character
            ):
                token_start += 1

            start_positions.append(token_start - 1)

            # Find the token containing the answer's final character
            token_end = context_end

            while (
                token_end >= context_start
                and offsets[token_end][1] >= end_character
            ):
                token_end -= 1

            end_positions.append(token_end + 1)

    tokenized_inputs["start_positions"] = start_positions
    tokenized_inputs["end_positions"] = end_positions

    return tokenized_inputs


print("Preprocessing function created successfully.")

Preprocessing function created successfully.


This cell is used to obtain the preprocessing function for processing SQuAD data for DistilBERT. The SQuAD dataset contains the correct answer in the character position within the context, while the model makes predictions based on the position of the tokens. Thus, this function transforms each question-context pair into a tokenized question-context pair and characterizes the answer's start and end positions in the character-level pair into token-level start and end positions. Also adds padding and truncation to ensure that all inputs are the same maximum length of 384 tokens.

## Preprocessing step to the training and validaiton datasets

In [13]:
tokenized_train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_validation_dataset = validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=validation_dataset.column_names
)

print("Training dataset processed successfully.")
print("Validation dataset processed successfully.")

print("\nProcessed training columns:")
print(tokenized_train_dataset.column_names)

print("\nNumber of processed training examples:")
print(len(tokenized_train_dataset))

print("\nNumber of processed validation examples:")
print(len(tokenized_validation_dataset))

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training dataset processed successfully.
Validation dataset processed successfully.

Processed training columns:
['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions']

Number of processed training examples:
10000

Number of processed validation examples:
2000


This cell performs the preprocessing function once for each training and validation example. Every question and context is turned into numerical token IDs, and the correct answer is turned into start/end token positions. The original text columns are deleted since the model needs numerical inputs, like input_ids, attention_mask, start_positions, and end_positions, when it is being trained.

## Loading the pretrained DistilBERT QA model

In [14]:
from transformers import (
    AutoModelForQuestionAnswering,
    DefaultDataCollator
)

# Load the pretrained DistilBERT model with a question-answering output layer
model = AutoModelForQuestionAnswering.from_pretrained(
    model_checkpoint
)

# Organizes individual examples into training batches
data_collator = DefaultDataCollator()

print("Question-answering model loaded successfully.")
print("Model type:", type(model).__name__)
print("Model parameters:", f"{model.num_parameters():,}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Question-answering model loaded successfully.
Model type: DistilBertForQuestionAnswering
Model parameters: 66,364,418


This cell loads the pretrained DistilBERT model that's been adapted to an extractive question answering version. Two output components in the model - one for the location of the start of the answer, and one for the end of the answer. Later, the data collator will merge each of these tokenized records into batches that can be efficiently processed while training.

## Training configuration

In [15]:
from transformers import TrainingArguments

output_directory = "./distilbert-squad-custom"

training_args = TrainingArguments(
    output_dir=output_directory,

    # Evaluate and save the model after every epoch
    eval_strategy="epoch",
    save_strategy="epoch",

    # Display training progress every 100 steps
    logging_strategy="steps",
    logging_steps=100,

    # Fine-tuning hyperparameters
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,

    # Keep only the two most recent checkpoints
    save_total_limit=2,

    # Restore the checkpoint with the lowest validation loss
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Use mixed-precision training on the Tesla T4 GPU
    fp16=True,

    # Disable external experiment-tracking services
    report_to="none",

    seed=42
)

print("Training configuration created successfully.")
print("Output directory:", training_args.output_dir)
print("Learning rate:", training_args.learning_rate)
print("Training epochs:", training_args.num_train_epochs)
print("Training batch size:", training_args.per_device_train_batch_size)

Training configuration created successfully.
Output directory: ./distilbert-squad-custom
Learning rate: 2e-05
Training epochs: 3
Training batch size: 8


The cell is used to fine-tune the model. We chose a very small learning rate of 2e-5, to update pretrained DistilBERT weights gradually; batch size of 8 to make sure the learning process fit within the memory of the T4 GPU; and 3 epochs to train the model for the SQuAD task without wasting time. The model will be assessed and stored at the end of every epoch and the checkpoint with the biggest loss on the validation set will be automatically loaded. HuggingFace provides TrainingArguments to manage parameters like batch size, training steps, evaluation, logging, and saving checkpoints.

## Hugging face trainer


In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer created successfully.")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))

Trainer created successfully.
Training examples: 10000
Validation examples: 2000


It creates the Hugging Face Trainer, it is responsible for connecting the DistilBERT model, training configuration, tokenizer, and data collator. The Trainer dynamically manages batching, forwad propagation, loss calc, backpropogatoin, parameter updates, validation.

## fine tuning distilBERT model

In [17]:
training_result = trainer.train()

print("\nTraining completed successfully.")
print("Training loss:", training_result.training_loss)
print("Training runtime:", training_result.metrics["train_runtime"])

Epoch,Training Loss,Validation Loss
1,1.758324,1.636381
2,1.247741,1.525573
3,0.927199,1.553253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed successfully.
Training loss: 1.5807195139567056
Training runtime: 372.0663


it starts the finetuning process. Trainer passes batches of SQuAD questions and contexts thru BERT, compare the predicted answer start and end positions with the correct positions, calculates the loss, and updates the model parameters thru backpropogation. Model eval is done after every epoch and the best checkpoint is selected subjective to lowest validation loss.

## Evlauting the best model

In [18]:
evaluation_results = trainer.evaluate()

print("Final evaluation results:")

for metric_name, metric_value in evaluation_results.items():
    print(f"{metric_name}: {metric_value}")

Training Loss,Validation Loss,Epoch
0.927199,1.525573,3


Final evaluation results:
eval_loss: 1.5255730152130127


Saiving the lowes loss version of the fine tuned bert model on the validation dataset. In my case lowest validation loss was seen in Epoch 2.

## Save the fine-tuned model and tokenizer

In [19]:
final_model_directory = "./final-distilbert-squad-qa"

# Save the fine-tuned model
trainer.save_model(final_model_directory)

# Save the tokenizer used during training
tokenizer.save_pretrained(final_model_directory)

print("Fine-tuned model and tokenizer saved successfully.")
print("Saved location:", final_model_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model and tokenizer saved successfully.
Saved location: ./final-distilbert-squad-qa


In [1]:
import torch
import transformers

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Transformers version: 4.57.1
PyTorch version: 2.11.0+cu128
GPU available: True


In [2]:
import os

final_model_directory = "./final-distilbert-squad-qa"

print("Model folder exists:", os.path.exists(final_model_directory))

if os.path.exists(final_model_directory):
    print("\nSaved files:")
    for filename in os.listdir(final_model_directory):
        print("-", filename)
else:
    print("The saved model folder was not found.")

Model folder exists: True

Saved files:
- tokenizer_config.json
- model.safetensors
- tokenizer.json
- config.json
- training_args.bin


## Creating QA pipeline

In [3]:
from transformers import pipeline

qa_pipeline = pipeline(
    task="question-answering",
    model=final_model_directory,
    tokenizer=final_model_directory,
    device=0
)

print("Custom question-answering pipeline created successfully.")

Device set to use cuda:0


Custom question-answering pipeline created successfully.


it loads the saved fine tuned bert model thru Hugging Face's pipeline() function. It automatically tokenizes the quesitons and context, and sends them thru the trained model, it predicts the most likely start and end positions of the answer, and converts the selected tokens into readable texts.

In [6]:
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    pipeline
)

# Reload the original DistilBERT tokenizer
qa_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased",
    use_fast=True
)

# Ensure the tokenizer sends only inputs supported by DistilBERT
qa_tokenizer.model_input_names = [
    "input_ids",
    "attention_mask"
]

# Load your locally fine-tuned model
qa_model = AutoModelForQuestionAnswering.from_pretrained(
    final_model_directory
)

# Recreate the question-answering pipeline
qa_pipeline = pipeline(
    task="question-answering",
    model=qa_model,
    tokenizer=qa_tokenizer,
    device=0
)

print("Question-answering pipeline fixed successfully.")
print("Tokenizer inputs:", qa_tokenizer.model_input_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


Question-answering pipeline fixed successfully.
Tokenizer inputs: ['input_ids', 'attention_mask']


This cell will re-load the DistilBERT tokenizer and retain your local question-answering model. We explicitly exclude token_type_ids from the model inputs to the tokenizer since DistilBERT needs input_ids and attention_mask only. This only fixes the inference pipeline—your trained model is the same.

## Testing the QA pipeline

In [7]:
context = """
Artificial intelligence is the field of creating computer systems that can
perform tasks that normally require human intelligence. Machine learning is
a subfield of artificial intelligence in which computer systems learn patterns
from data. Deep learning is a branch of machine learning that uses neural
networks with multiple layers.
"""

question = "What does deep learning use?"

result = qa_pipeline(
    question=question,
    context=context
)

print("Question:", question)
print("Answer:", result["answer"])
print("Confidence score:", round(result["score"], 4))
print("Answer start position:", result["start"])
print("Answer end position:", result["end"])

Question: What does deep learning use?
Answer: neural
networks with multiple layers
Confidence score: 0.0511
Answer start position: 298
Answer end position: 334


## testing with multiple questions

In [8]:
test_context = """
Artificial intelligence is the field of creating computer systems that can
perform tasks that normally require human intelligence. Machine learning is
a subfield of artificial intelligence in which computer systems learn patterns
from data. Deep learning is a branch of machine learning that uses neural
networks with multiple layers.
"""

test_questions = [
    "What is machine learning?",
    "What does deep learning use?",
    "What is artificial intelligence?"
]

for question in test_questions:
    result = qa_pipeline(
        question=question,
        context=test_context
    )

    print("=" * 70)
    print("Question:", question)
    print("Predicted answer:", result["answer"])
    print("Confidence score:", round(result["score"], 4))
    print("Start position:", result["start"])
    print("End position:", result["end"])

Question: What is machine learning?
Predicted answer: computer systems learn patterns
from data
Confidence score: 0.0673
Start position: 199
End position: 240
Question: What does deep learning use?
Predicted answer: neural
networks with multiple layers
Confidence score: 0.0511
Start position: 298
End position: 334
Question: What is artificial intelligence?
Predicted answer: Machine learning
Confidence score: 0.1167
Start position: 132
End position: 148


## Creating and saving a results table

In [9]:
import pandas as pd

expected_answers = [
    "a subfield of artificial intelligence",
    "neural networks with multiple layers",
    "the field of creating computer systems that can perform tasks that normally require human intelligence"
]

results_list = []

for question, expected_answer in zip(test_questions, expected_answers):
    prediction = qa_pipeline(
        question=question,
        context=test_context
    )

    results_list.append({
        "Question": question,
        "Expected Answer": expected_answer,
        "Predicted Answer": prediction["answer"],
        "Confidence Score": round(prediction["score"], 4),
        "Start Position": prediction["start"],
        "End Position": prediction["end"]
    })

results_table = pd.DataFrame(results_list)

print("Question-answering results:")
display(results_table)

results_table.to_csv(
    "question_answering_results.csv",
    index=False
)

print("\nResults saved as question_answering_results.csv")

Question-answering results:


,Question,Expected Answer,Predicted Answer,Confidence Score,Start Position,End Position
0,What is machine learning?,a subfield of artificial intelligence,computer systems learn patterns\nfrom data,0.0673,199,240
1,What does deep learning use?,neural networks with multiple layers,neural\nnetworks with multiple layers,0.0511,298,334
2,What is artificial intelligence?,the field of creating computer systems that ca...,Machine learning,0.1167,132,148



Results saved as question_answering_results.csv


In [10]:
clear_context = """
Canada is a country located in North America. Ottawa is the capital city of
Canada. English and French are the official languages of Canada. The country
has ten provinces and three territories.
"""

clear_questions = [
    "What is the capital city of Canada?",
    "What are the official languages of Canada?",
    "How many provinces does Canada have?"
]

for question in clear_questions:
    prediction = qa_pipeline(
        question=question,
        context=clear_context
    )

    print("=" * 70)
    print("Question:", question)
    print("Predicted answer:", prediction["answer"])
    print("Confidence score:", round(prediction["score"], 4))
    print("Start position:", prediction["start"])
    print("End position:", prediction["end"])

Question: What is the capital city of Canada?
Predicted answer: Ottawa
Confidence score: 0.9659
Start position: 47
End position: 53
Question: What are the official languages of Canada?
Predicted answer: English and French
Confidence score: 0.6568
Start position: 85
End position: 103
Question: How many provinces does Canada have?
Predicted answer: ten
Confidence score: 0.2586
Start position: 158
End position: 161
